# Tarea 3: Análisis del rendimiento de SIGA durante la inscripción de asignaturas

Este notebook acompaña el análisis de la Tarea 3. El sistema estudiado es hipotético y se usa con fines académicos. Los parámetros no representan mediciones reales de SIGA.

## Integrantes

Corresponde a la sección Formato y entrega del enunciado, donde se piden los nombres y roles.

- TODO: Formato y entrega. Nombre completo y rol USM del integrante 1.
- TODO: Formato y entrega. Nombre completo y rol USM del integrante 2, si corresponde.

## Objetivos

Completar un simulador de una fila M/M/1/K con llegadas de Poisson, servicio exponencial, un servidor, una fila FCFS y capacidad de 20 solicitudes. Calcular métricas experimentales, compararlas con las métricas teóricas de M/M/1 y de M/M/1/K, y observar la acumulación durante la sobrecarga y la recuperación.

## 1. Parámetros y métricas teóricas

Corresponde a las secciones 1.1, 2.1, 2.2 y 3.2 del enunciado.

La capacidad del sistema es de 20 solicitudes, incluida la que está en servicio. Las solicitudes que llegan cuando el sistema está lleno se rechazan.

Complete en la sección Parámetros y métricas teóricas las funciones `metricas_mm1` y `metricas_mm1k`, con las expresiones vistas en clase. Las métricas teóricas se comparan con las observadas más adelante.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import deque
from pathlib import Path
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display

np.set_printoptions(precision=4, suppress=True)

mu = 10.0  # solicitudes por segundo
K = 20  # capacidad total del sistema, incluida la solicitud en servicio
# Las métricas observadas se calculan entre el calentamiento y el horizonte, es decir, después del arranque del sistema.
horizonte = 10000.0  # segundos
calentamiento = 1000.0  # segundos
# Carpeta con los registros. 
RUTA_DATOS = "datos" # Cambiar por ruta de carpeta datos/ si corresponde 

# Cada escenario usa su propio registro de llegadas y servicios. El valor lambda es la tasa media de llegadas y sirve como referencia.
escenarios = [
    {"codigo": "A", "descripcion": "Operación habitual", "lambda": 6.0, "archivo": "escenario_A.csv"},
    {"codigo": "B", "descripcion": "Inicio de inscripción", "lambda": 8.0, "archivo": "escenario_B.csv"},
    {"codigo": "C", "descripcion": "Alta congestión", "lambda": 9.5, "archivo": "escenario_C.csv"},
    {"codigo": "D", "descripcion": "Sobrecarga durante inscripción", "lambda": 10.5, "archivo": "escenario_D.csv"},
]


def metricas_mm1(tasa_llegada, tasa_servicio):
    """Métricas teóricas de M/M/1 con capacidad infinita.

    Devuelva un diccionario con las claves "rho", "estable", "L", "Lq", "W" y "Wq".
    Use None para las métricas que no existan.
    """
    # TODO: 1.1. Complete las métricas de M/M/1 con las expresiones vistas en clase.
    return None


def metricas_mm1k(tasa_llegada, tasa_servicio, capacidad):
    """Métricas teóricas de M/M/1/K con capacidad finita.

    Devuelva un diccionario con las claves "rho", "pi0", "P_B", "lambda_e", "L", "Lq", "W", "Wq" y "ocupacion".
    La clave "ocupacion" corresponde a 1 - pi0.
    """
    # TODO: 2.1 y 2.2. Complete las métricas de M/M/1/K con las expresiones vistas en clase.
    return None


teoricas_mm1 = {e["codigo"]: metricas_mm1(e["lambda"], mu) for e in escenarios}
teoricas_mm1k = {e["codigo"]: metricas_mm1k(e["lambda"], mu, K) for e in escenarios}

print(f"Tasa media de servicio: {mu:g} solicitudes/s. Capacidad del sistema: {K} solicitudes.")
print("Escenario | lambda | rho | P_B (M/M/1/K) | lambda_e (M/M/1/K) | L (M/M/1/K) | W (M/M/1/K) [s]")
for e in escenarios:
    t = teoricas_mm1k[e["codigo"]]
    print(f"{e['codigo']} | {e['lambda']:.1f} | {t['rho']:.2f} | {t['P_B']:.4f} | {t['lambda_e']:.4f} | {t['L']:.4f} | {t['W']:.4f}")

## 2. Registros de llegadas y servicios

Corresponde a la introducción de la sección 3 del enunciado.

Cada escenario tiene un registro fijo en la carpeta `datos`, con una fila por solicitud. La columna `llegada` indica el instante de llegada, en segundos, y la columna `servicio` indica la duración de su servicio, en segundos. Si la solicitud se rechaza, su duración de servicio no se usa.

La función `cargar_registro` lee un registro y devuelve las columnas como arrays de NumPy.

In [ ]:
# Cada fila del registro es una solicitud, en orden de llegada. Primera columna: instante de llegada. Segunda columna: duración del servicio. Ambas en segundos.
def cargar_registro(ruta):
    """Lee un registro con las columnas llegada y servicio."""
    datos = np.loadtxt(ruta, delimiter=",", skiprows=1)
    return datos[:, 0], datos[:, 1]


## 3. Simulación de la fila FCFS con capacidad finita

Corresponde a la sección 3.1 del enunciado.

Hay un solo servidor, una fila FCFS y una capacidad finita. Al llegar una solicitud, se cuentan las solicitudes admitidas que todavía están en el sistema, es decir, las que no han salido en ese instante. Si el sistema está lleno, la solicitud se rechaza y no cambia el estado. Si no está lleno, la solicitud entra y se atiende en orden FCFS. Las solicitudes rechazadas no reciben servicio y no entran en los promedios de $W$ y $W_q$, pero sí en la estimación de $P_B$.

In [ ]:
def simular_fcfs_capacidad(instantes_llegada, duraciones_servicio, capacidad):
    """Simula una fila FCFS con un servidor y capacidad total.

    Cada solicitud tiene un instante de llegada y una duración de servicio. Las solicitudes se revisan en orden de llegada.
    Las solicitudes rechazadas no reciben servicio.
    """
    instantes_llegada = np.asarray(instantes_llegada, dtype=float)
    duraciones_servicio = np.asarray(duraciones_servicio, dtype=float)

    # es_admitida[i] es True si la solicitud i entra al sistema. Al inicio, ninguna ha entrado.
    es_admitida = np.zeros(len(instantes_llegada), dtype=bool)
    # Una entrada por cada solicitud admitida, en orden de llegada.
    instantes_inicio = []
    instantes_salida = []
    # Instantes de salida de las solicitudes admitidas que todavía están en el sistema.
    # El largo de esta lista es el número de solicitudes en el sistema en el instante de una llegada.
    salidas_pendientes = deque()
    # Instante en que termina el servicio de la última solicitud admitida. Al inicio el sistema está vacío.
    instante_salida_anterior = 0.0

    for i in range(len(instantes_llegada)):
        instante_llegada_i = instantes_llegada[i]
        duracion_servicio_i = duraciones_servicio[i]

        # Antes de revisar la solicitud i, se quitan las solicitudes que ya salieron antes de su llegada.
        while salidas_pendientes and salidas_pendientes[0] <= instante_llegada_i:
            salidas_pendientes.popleft()

        # TODO: 3.1 (paso 1). Asigne a entra_i si la solicitud i entra al sistema.
        # Regla: entra solo si el sistema no está lleno, es decir, si hay menos solicitudes en el sistema que la capacidad.
        entra_i = None
        if not entra_i:
            continue  # la solicitud es rechazada y no cambia el estado del sistema
        es_admitida[i] = True

        # TODO: 3.1 (paso 2). Asigne a instante_inicio_i el instante en que empieza el servicio de la solicitud i.
        # Regla: si el servidor está libre al llegar, empieza en ese instante. Si no, empieza cuando termina el servicio de la última solicitud admitida.
        instante_inicio_i = None

        # TODO: 3.1 (paso 3). Asigne a instante_salida_i el instante en que termina el servicio de la solicitud i.
        # Regla: el servicio dura duracion_servicio_i, así que la salida es el inicio más esa duración.
        instante_salida_i = None

        instantes_inicio.append(instante_inicio_i)
        instantes_salida.append(instante_salida_i)
        salidas_pendientes.append(instante_salida_i)
        instante_salida_anterior = instante_salida_i

    # Solo las solicitudes admitidas tienen instantes de inicio y de salida.
    return {
        "llegadas": instantes_llegada[es_admitida],  # llegadas de las admitidas
        "inicios": np.asarray(instantes_inicio, dtype=float),  # inicios de servicio de las admitidas
        "salidas": np.asarray(instantes_salida, dtype=float),  # salidas de las admitidas
        "servicios": duraciones_servicio[es_admitida],  # duraciones de servicio de las admitidas
        "rechazada": ~es_admitida,  # True para las rechazadas, entre todas las llegadas
        "llegadas_totales": instantes_llegada,  # llegadas de todas las solicitudes
    }


## 4. Métricas experimentales

Corresponde a la sección 3.2 del enunciado.

La función auxiliar `promedio_temporal_conteo` calcula promedios temporales a partir de los instantes de cambio. Complete `metricas_experimentales` con las siguientes definiciones, todas calculadas dentro de la ventana de observación $[\text{calentamiento}, \text{horizonte}]$:

- $L$ y $\rho$ observada: promedio temporal de $N(t)$ y fracción del tiempo con el servidor ocupado.
- $L_q$: promedio temporal de las solicitudes en espera. Use la relación entre $L$ y la fracción de tiempo ocupado.
- $W$ y $W_q$: promedios de las solicitudes admitidas que llegan dentro de la ventana. Incluya las que salen después del horizonte.
- $P_B$ observada: fracción de las llegadas dentro de la ventana que se rechazan.
- $\lambda_e$ observada: solicitudes admitidas por unidad de tiempo dentro de la ventana.

Compare las métricas observadas con las teóricas de M/M/1 y de M/M/1/K en los cuatro escenarios. Si una métrica teórica no es válida en algún escenario, indíquelo y compare la simulación con lo que sí se puede calcular.


In [ ]:
# Promedio en el tiempo de un conteo N(t): área bajo la curva dividida por la duración de la ventana (presentación 07, sección El área bajo N(t)).
def promedio_temporal_conteo(eventos, cambios, inicio, fin):
    """Promedia un conteo por tiempo usando sus instantes de cambio."""
    eventos = np.asarray(eventos, dtype=float)
    cambios = np.asarray(cambios, dtype=float)
    if eventos.size == 0:
        return 0.0

    # Los cambios se ordenan por instante. Cada llegada suma 1 al conteo y cada salida resta 1.
    orden = np.argsort(eventos, kind="mergesort")
    eventos = eventos[orden]
    cambios = cambios[orden]
    indice = np.searchsorted(eventos, inicio, side="right")
    conteo = float(np.sum(cambios[:indice]))
    instante_previo = float(inicio)
    area = 0.0

    for j in range(indice, len(eventos)):
        instante = eventos[j]
        if instante >= fin:
            break
        # Entre dos cambios consecutivos el conteo no varía, así que el área de ese tramo es conteo por duración.
        area += conteo * (instante - instante_previo)
        conteo += cambios[j]
        instante_previo = instante

    area += conteo * (fin - instante_previo)
    return area / (fin - inicio)


# Métricas observadas dentro de la ventana [calentamiento, horizonte]. Las rechazadas no reciben servicio, pero sí cuentan para P_B.
def metricas_experimentales(datos, inicio_observacion, fin_observacion):
    llegadas = datos["llegadas"]
    inicios = datos["inicios"]
    salidas = datos["salidas"]
    llegadas_totales = datos["llegadas_totales"]
    rechazada = datos["rechazada"]

    # TODO: 3.2 (paso 4). Calcule L con promedio_temporal_conteo, usando llegadas (+1) y salidas (-1).
    # Vea trayectoria_N en la sección Visualización del sistema, que construye eventos y cambios de la misma forma.
    L = None
    # TODO: 3.2 (paso 5). Calcule la fracción de la ventana con el servidor ocupado, usando [inicios, salidas].
    rho_ocupacion = None
    # TODO: 3.2 (paso 6). Calcule Lq a partir de L y rho_ocupacion.
    Lq = None
    # TODO: 3.2 (paso 7). Calcule W y Wq para las solicitudes admitidas que llegan dentro de la ventana.
    W = None
    Wq = None
    # TODO: 3.2 (paso 8). Calcule P_B y lambda_e para las llegadas dentro de la ventana.
    P_B = None
    lambda_e = None

    return {
        "L": L, "Lq": Lq, "rho": rho_ocupacion, "W": W, "Wq": Wq,
        "P_B": P_B, "lambda_e": lambda_e,
    }

## 5. Ejecución de los cuatro escenarios

Corresponde a la sección 3.2 del enunciado.

Cada escenario usa su registro de llegadas y servicios, con el horizonte y el calentamiento definidos al inicio del notebook. Las solicitudes que siguen en el sistema al final se mantienen en los promedios temporales. Las solicitudes que salen después del horizonte se incluyen en $W$ y $W_q$.

In [ ]:
# Cada escenario se simula con su registro. Las métricas teóricas, definidas al inicio del notebook, se muestran junto a las observadas.
resultados = []
for escenario in escenarios:
    llegadas, servicios = cargar_registro(f"{RUTA_DATOS}/{escenario['archivo']}")
    datos = simular_fcfs_capacidad(llegadas, servicios, K)
    datos["codigo"] = escenario["codigo"]
    datos["descripcion"] = escenario["descripcion"]
    datos["lambda"] = escenario["lambda"]
    datos["teoricas_mm1k"] = teoricas_mm1k[escenario["codigo"]]
    datos["teoricas_mm1"] = teoricas_mm1[escenario["codigo"]]
    # Solo cuenta la ventana posterior al calentamiento. Las solicitudes que salen después del horizonte sí entran en W y W_q.
    datos["experimentales"] = metricas_experimentales(datos, calentamiento, horizonte)
    resultados.append(datos)


def mostrar_valor(valor):
    return "No aplica" if valor is None else f"{valor:.4f}"


def tabla_html(titulo, encabezados, filas):
    """Muestra una tabla con formato HTML, sin bibliotecas adicionales."""
    estilo = (
        "<style>"
        ".tabla-tel {border-collapse: collapse; font-size: 13px;}"
        ".tabla-tel th, .tabla-tel td {padding: 4px 12px; border-bottom: 1px solid #d1d5db;}"
        ".tabla-tel th {background: #f3f4f6; text-align: right;}"
        ".tabla-tel td {text-align: right;}"
        ".tabla-tel th:first-child, .tabla-tel td:first-child {text-align: left;}"
        "</style>"
    )
    cabecera = "".join(f"<th>{texto}</th>" for texto in encabezados)
    cuerpo = "".join("<tr>" + "".join(f"<td>{celda}</td>" for celda in fila) + "</tr>" for fila in filas)
    display(HTML(f"{estilo}<p><b>{titulo}</b></p><table class='tabla-tel'><tr>{cabecera}</tr>{cuerpo}</table>"))


tabla_html(
    "Tabla 1. Ocupación, bloqueo y tasa admitida en M/M/1/K",
    ["Escenario", "1 - π0 teórica", "ρ observada", "P_B teórica", "P_B simulada", "λe teórica", "λe simulada"],
    [[d["codigo"], mostrar_valor(d["teoricas_mm1k"]["ocupacion"]), mostrar_valor(d["experimentales"]["rho"]),
      mostrar_valor(d["teoricas_mm1k"]["P_B"]), mostrar_valor(d["experimentales"]["P_B"]),
      mostrar_valor(d["teoricas_mm1k"]["lambda_e"]), mostrar_valor(d["experimentales"]["lambda_e"])]
     for d in resultados],
)

metricas = [("L", "L (solicitudes)"), ("Lq", "Lq (solicitudes)"), ("W", "W (s)"), ("Wq", "Wq (s)")]
tabla_html(
    "Tabla 2. Métricas de solicitudes admitidas: M/M/1, M/M/1/K y simulación",
    ["Escenario", "Métrica", "M/M/1", "M/M/1/K", "Simulación"],
    [[d["codigo"], nombre, mostrar_valor(d["teoricas_mm1"][clave]), mostrar_valor(d["teoricas_mm1k"][clave]),
      mostrar_valor(d["experimentales"][clave])]
     for d in resultados for clave, nombre in metricas],
)


## 6. Visualización del sistema

Corresponde a la sección 3.2 del enunciado.

La rutina siguiente construye $N(t)$ desde las llegadas y salidas admitidas. La línea roja marca la capacidad del sistema. El gráfico de métricas compara la simulación con M/M/1/K en la probabilidad de bloqueo, y con M/M/1 y M/M/1/K en el número medio de solicitudes. Si una métrica de M/M/1 no existe en algún escenario, el gráfico lo indica.

In [ ]:
# N(t) sube en 1 cada vez que entra una solicitud admitida y baja en 1 cada vez que sale una.
def trayectoria_N(datos, tiempo_final):
    llegadas = datos["llegadas"]
    salidas = datos["salidas"]
    llegadas_visibles = llegadas[llegadas < tiempo_final]
    salidas_visibles = salidas[salidas < tiempo_final]
    eventos = np.concatenate((llegadas_visibles, salidas_visibles))
    cambios = np.concatenate((np.ones(len(llegadas_visibles)), -np.ones(len(salidas_visibles))))
    if len(eventos) == 0:
        return np.array([0.0, tiempo_final]), np.array([0.0, 0.0])
    orden = np.argsort(eventos, kind="mergesort")
    eventos = eventos[orden]
    conteos = np.cumsum(cambios[orden])
    tiempos_grafico = np.concatenate(([0.0], eventos, [tiempo_final]))
    valores_grafico = np.concatenate(([0.0], conteos, [conteos[-1]]))
    return tiempos_grafico, valores_grafico


fig, axes = plt.subplots(4, 1, figsize=(10, 10), sharex=True)
for ax, datos in zip(axes, resultados):
    tiempos_grafico, valores_grafico = trayectoria_N(datos, horizonte)
    ax.step(tiempos_grafico, valores_grafico, where="post", linewidth=0.9)
    ax.axhline(K, color="tab:red", linestyle="--", linewidth=0.9, label=f"K = {K}")
    ax.set_xlim(0, horizonte)
    ax.set_ylim(0, K + 2)
    ax.set_ylabel("N(t)")
    ax.set_title(f"Escenario {datos['codigo']}: lambda = {datos['lambda']:g} solicitudes/s, rho = {datos['teoricas_mm1k']['rho']:.2f}", loc="left")
    ax.legend(loc="upper right", fontsize=8)
axes[-1].set_xlabel("Tiempo de simulación [s]")
plt.tight_layout()
plt.show()

codigos = [d["codigo"] for d in resultados]
x = np.arange(len(resultados))
ancho = 0.38
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.bar(x - ancho / 2, [d["teoricas_mm1k"]["P_B"] for d in resultados], ancho, label="M/M/1/K teórico")
ax1.bar(x + ancho / 2, [d["experimentales"]["P_B"] for d in resultados], ancho, label="Simulación")
ax1.set_xticks(x, codigos)
ax1.set_ylabel("P_B")
ax1.set_title("Probabilidad de bloqueo")
ax1.legend()
# Si L de M/M/1 no existe en un escenario, no se dibuja la barra y se indica con un texto.
ancho_l = 0.27
l_mm1 = [np.nan if d["teoricas_mm1"]["L"] is None else d["teoricas_mm1"]["L"] for d in resultados]
ax2.bar(x - ancho_l, l_mm1, ancho_l, label="M/M/1 teórico")
ax2.bar(x, [d["teoricas_mm1k"]["L"] for d in resultados], ancho_l, label="M/M/1/K teórico")
ax2.bar(x + ancho_l, [d["experimentales"]["L"] for d in resultados], ancho_l, label="Simulación")
for xi, valor in zip(x, l_mm1):
    if np.isnan(valor):
        ax2.text(xi - ancho_l, 0.3, "No aplica", rotation=90, ha="center", va="bottom", fontsize=8)
ax2.set_xticks(x, codigos)
ax2.set_ylabel("L")
ax2.set_title("Solicitudes medias en el sistema")
ax2.legend()
plt.tight_layout()
plt.show()

### Animación de la fila

Corresponde a la sección 3.2 del enunciado.

La animación muestra la fila, el servidor y el sistema de cada escenario. El sistema incluye la fila y el servidor, y su capacidad incluye la solicitud en servicio. Las llegadas rechazadas se marcan en rojo.

Para cambiar de escenario, modifique la variable escenario_animado en la celda de la animación.

Advertencia: la animación genera un archivo pesado. Al ejecutar la celda, espere a que aparezca completa. Si el notebook se vuelve lento, reduzca el número de solicitudes de la variable solicitudes_animadas.

In [ ]:
# Escenario que se anima. Cambie la letra para ver otro escenario: "A", "B", "C" o "D".
escenario_animado = "B"
# Número de solicitudes admitidas que se animan en cada escenario. Una ventana más larga produce una animación más pesada.
solicitudes_animadas = {"A": 80, "B": 80, "C": 80, "D": 170}
# Ancho de la animación respecto del ancho disponible del notebook.
ancho_animacion = "80%"
# Todas las animaciones tienen la misma duración: 350 fotogramas a 8 por segundo, unos 44 s.
fotogramas_totales = 350
fps_animacion = 8


def animar_fila(datos, capacidad, max_solicitudes=80):
    """Anima el sistema para las primeras solicitudes admitidas de un escenario."""
    n = min(max_solicitudes, len(datos["llegadas"]))
    llegadas = datos["llegadas"][:n]
    inicios = datos["inicios"][:n]
    salidas = datos["salidas"][:n]
    # Número de cada solicitud en el registro completo, incluidas las rechazadas.
    ids_admitidas = np.flatnonzero(~datos["rechazada"])[:n]
    llegadas_todas = datos["llegadas_totales"]
    # Solo se muestran los rechazos ocurridos hasta la última llegada admitida de la ventana.
    ids_rechazadas = np.flatnonzero(datos["rechazada"] & (llegadas_todas <= llegadas[-1]))
    tiempos_rechazos = llegadas_todas[ids_rechazadas]
    cuadros = np.unique(np.concatenate(([0.0], llegadas, salidas, tiempos_rechazos)))
    # Si el escenario tiene menos eventos, su último estado se mantiene hasta completar fotogramas_totales.
    n_eventos = len(cuadros)
    if n_eventos < fotogramas_totales:
        cuadros = np.concatenate((cuadros, np.full(fotogramas_totales - n_eventos, cuadros[-1])))

    # La capacidad incluye la solicitud en servicio, así que la fila tiene un puesto menos.
    puestos_fila = capacidad - 1
    paso = 0.55  # separación entre puestos de la fila
    x_primer_puesto = 11.6  # el puesto junto al servidor es el primero en ocuparse

    fig, ax = plt.subplots(figsize=(12, 4.4), dpi=90)
    fig.subplots_adjust(left=0, right=1, bottom=0, top=1)
    ax.set_xlim(-0.6, 16.0)
    ax.set_ylim(-1.5, 2.4)
    ax.axis("off")

    # Fuera del sistema: llegadas y salidas.
    ax.text(-0.5, 0.78, "Llegadas", ha="left", fontsize=13)
    ax.annotate("", xy=(1.1, 0.1), xytext=(-0.4, 0.1), arrowprops={"arrowstyle": "->", "lw": 1.8})
    ax.text(14.5, 0.1, "Salidas", ha="left", va="center", fontsize=13)
    ax.annotate("", xy=(14.4, 0.1), xytext=(13.5, 0.1), arrowprops={"arrowstyle": "->", "lw": 1.8})

    # El recuadro punteado es el sistema. Dentro están la fila y el servidor.
    sistema = plt.Rectangle((0.6, -1.15), 13.4, 2.5, facecolor="none", edgecolor="#6b7280", linewidth=2, linestyle="--")
    ax.add_patch(sistema)
    # Al terminar la ventana, el sistema se atenúa y aparece un rótulo al centro.
    atenuado = plt.Rectangle((0.6, -1.15), 13.4, 2.5, facecolor="#9ca3af", alpha=0.0, edgecolor="none", zorder=2.5)
    ax.add_patch(atenuado)
    rotulo_fin = ax.text(7.3, 0.1, "Fin de la ventana de animación", ha="center", va="center", fontsize=15,
                         fontweight="bold", color="#1f2937", zorder=6, visible=False,
                         bbox={"boxstyle": "round,pad=0.6", "facecolor": "white", "edgecolor": "#6b7280", "linewidth": 2})
    ax.text(0.8, 1.1, f"Sistema (capacidad {capacidad})", ha="left", fontsize=12)
    texto_lleno = ax.text(13.8, 1.1, "Sistema lleno", ha="right", fontsize=12, color="tab:red", visible=False)

    # La fila: cada puesto vacío se dibuja como un cuadro. Se llena de derecha a izquierda.
    ax.add_patch(plt.Rectangle((1.1, -0.6), 10.85, 1.3, facecolor="#e8f3fb", edgecolor="#2f6f9f", linewidth=1.5))
    ax.text(6.5, 0.82, f"Fila FCFS ({puestos_fila} puestos de espera)", ha="center", fontsize=12)
    for j in range(puestos_fila):
        x = x_primer_puesto - j * paso
        ax.add_patch(plt.Rectangle((x - 0.23, -0.2), 0.46, 0.6, facecolor="white", edgecolor="#9ec5e0", linewidth=1))

    # El servidor cambia de color cuando atiende una solicitud.
    servidor = plt.Rectangle((12.2, -0.35), 1.3, 0.9, facecolor="white", edgecolor="black", linewidth=2)
    ax.add_patch(servidor)
    ax.text(12.85, 0.68, "Servidor", ha="center", fontsize=12)

    # Cada solicitud conserva el mismo color en la fila y en el servidor.
    cola = ax.scatter([], [], s=330, color="tab:blue", edgecolor="black", zorder=3)
    solicitud_servidor = ax.scatter([], [], s=480, color="tab:blue", edgecolor="black", zorder=4)
    rechazo = ax.scatter([], [], marker="x", s=360, color="tab:red", linewidths=3, zorder=5)
    etiquetas_cola = [ax.text(0, -0.4, "", ha="center", fontsize=9) for _ in range(puestos_fila)]
    etiqueta_servidor = ax.text(12.85, -0.65, "", ha="center", fontsize=10)
    etiqueta_rechazo = ax.text(0.1, -0.5, "Rechazada", ha="center", fontsize=11, color="tab:red", visible=False)
    texto_estado = ax.text(0.6, 1.72, "", ha="left", fontsize=12)
    texto_evento = ax.text(0.6, -1.4, "", ha="left", fontsize=11)
    # Título con el escenario que se está animando.
    ax.text(0.6, 2.22, f"Escenario {datos['codigo']}: {datos['descripcion']}", ha="left", fontsize=13, fontweight="bold")

    def actualizar(indice):
        tiempo = cuadros[indice]
        en_fila = np.flatnonzero((llegadas <= tiempo) & (inicios > tiempo))
        en_servicio = np.flatnonzero((inicios <= tiempo) & (tiempo < salidas))
        completadas = int(np.sum(salidas <= tiempo))
        en_sistema = int(np.sum(llegadas <= tiempo)) - completadas
        rechazadas = int(np.sum(tiempos_rechazos <= tiempo))

        # La primera solicitud en espera ocupa el puesto junto al servidor y las demás se ubican hacia la izquierda.
        posiciones_x = [x_primer_puesto - paso * j for j in range(len(en_fila))]
        cola.set_offsets(np.array([[x, 0.1] for x in posiciones_x], dtype=float).reshape(-1, 2))
        for j, etiqueta in enumerate(etiquetas_cola):
            if j < len(en_fila):
                etiqueta.set_position((posiciones_x[j], -0.4))
                etiqueta.set_text(str(ids_admitidas[en_fila[j]] + 1))
                etiqueta.set_visible(True)
            else:
                etiqueta.set_text("")
                etiqueta.set_visible(False)

        # El servidor cambia de color, pero la solicitud que atiende conserva su color.
        if len(en_servicio) == 1:
            id_servicio = int(en_servicio[0])
            solicitud_servidor.set_offsets([[12.85, 0.1]])
            etiqueta_servidor.set_text(f"Solicitud {ids_admitidas[id_servicio] + 1}")
            servidor.set_facecolor("tab:orange")
        else:
            solicitud_servidor.set_offsets(np.empty((0, 2)))
            etiqueta_servidor.set_text("")
            servidor.set_facecolor("white")

        # Cuando el sistema está lleno, su borde se vuelve rojo.
        lleno = en_sistema >= capacidad
        sistema.set_edgecolor("tab:red" if lleno else "#6b7280")
        sistema.set_linewidth(3 if lleno else 2)
        texto_lleno.set_visible(lleno)

        # Una llegada rechazada muestra una marca roja en la entrada del sistema.
        rechazos_ahora = np.flatnonzero(np.isclose(tiempos_rechazos, tiempo, rtol=0, atol=1e-10))
        hay_rechazo = len(rechazos_ahora) > 0
        rechazo.set_offsets([[0.1, 0.1]] if hay_rechazo else np.empty((0, 2)))
        etiqueta_rechazo.set_visible(hay_rechazo)

        llegadas_evento = np.flatnonzero(np.isclose(llegadas, tiempo, rtol=0, atol=1e-10))
        salidas_evento = np.flatnonzero(np.isclose(salidas, tiempo, rtol=0, atol=1e-10))
        partes_evento = []
        if hay_rechazo:
            partes_evento.append("llegada rechazada de solicitud " + str(int(ids_rechazadas[rechazos_ahora[0]]) + 1))
        if len(llegadas_evento):
            partes_evento.append("llegada de solicitud " + str(int(ids_admitidas[llegadas_evento[0]]) + 1))
        if len(salidas_evento):
            partes_evento.append("salida de solicitud " + str(int(ids_admitidas[salidas_evento[0]]) + 1))
        evento = ", ".join(partes_evento) if partes_evento else "sistema vacío"
        if indice >= n_eventos:
            evento = "fin de la ventana"
        # Cuando termina la ventana, el sistema se atenúa y aparece el rótulo de fin.
        fin_ventana = indice >= n_eventos
        atenuado.set_alpha(0.3 if fin_ventana else 0.0)
        rotulo_fin.set_visible(fin_ventana)
        texto_estado.set_text(
            f"Tiempo: {tiempo:.2f} s     En el sistema: {en_sistema} de {capacidad}     "
            f"Completadas: {completadas}     Rechazadas: {rechazadas}"
        )
        texto_evento.set_text(f"Evento: {evento}")
        return (cola, solicitud_servidor, servidor, sistema, rechazo, texto_lleno, etiqueta_rechazo,
                texto_estado, texto_evento, etiqueta_servidor, *etiquetas_cola)

    animacion = FuncAnimation(fig, actualizar, frames=len(cuadros), interval=120, blit=False, repeat=False)
    html_animacion = animacion.to_jshtml(fps=fps_animacion)
    # La animación ocupa el ancho indicado en ancho_animacion. La imagen y el deslizador se ajustan a ese ancho.
    estilo = (
        "<style>"
        ".anim-ancho .animation {display: block;}"
        ".anim-ancho .animation img {width: 100%; height: auto;}"
        ".anim-ancho input[type=range].anim-slider {width: 100%;}"
        "</style>"
    )
    display(HTML(f'{estilo}<div class="anim-ancho" style="width: {ancho_animacion};">{html_animacion}</div>'))
    plt.close(fig)
    return animacion


resultado_animado = next(datos for datos in resultados if datos["codigo"] == escenario_animado)
animacion_fila = animar_fila(resultado_animado, capacidad=K, max_solicitudes=solicitudes_animadas[escenario_animado])


## 7. Experimento de sobrecarga y recuperación

Corresponde a la sección 3.3 del enunciado.

La gráfica de $N(t)$ muestra las dos etapas del registro `recuperacion.csv`, separadas por una línea punteada. La tasa media de servicio y la capacidad son las mismas de los escenarios anteriores.

In [ ]:
duracion_sobrecarga = 600.0
duracion_recuperacion = 600.0

# Un solo registro con las dos etapas. La fila no se reinicia en el cambio de tasa.
llegadas_recuperacion, servicios_recuperacion = cargar_registro(f"{RUTA_DATOS}/recuperacion.csv")
datos_recuperacion = simular_fcfs_capacidad(llegadas_recuperacion, servicios_recuperacion, K)

# Instante del cambio de tasa. Las solicitudes que quedan en el sistema en ese instante siguen atendiéndose después.
tiempo_cambio = duracion_sobrecarga
en_sistema_cambio = int(np.sum(datos_recuperacion["llegadas"] <= tiempo_cambio) - np.sum(datos_recuperacion["salidas"] <= tiempo_cambio))
rechazadas_sobrecarga = int(np.sum(datos_recuperacion["rechazada"] & (datos_recuperacion["llegadas_totales"] <= tiempo_cambio)))
print(f"Solicitudes en el sistema al cambiar la tasa: {en_sistema_cambio}")
print(f"Solicitudes rechazadas durante la sobrecarga: {rechazadas_sobrecarga}")

tiempos_rec, conteos_rec = trayectoria_N(datos_recuperacion, duracion_sobrecarga + duracion_recuperacion)
fig, ax = plt.subplots(figsize=(9, 4))
ax.step(tiempos_rec, conteos_rec, where="post", color="tab:purple")
ax.axvline(tiempo_cambio, color="black", linestyle="--", label="cambio de tasa")
ax.axhline(K, color="tab:red", linestyle=":", label="capacidad del sistema")
ax.set_xlabel("Tiempo [s]")
ax.set_ylabel("N(t)")
ax.set_title("Experimento de sobrecarga y recuperación")
ax.legend()
plt.tight_layout()
plt.show()
